# 01 — Database Setup & Schema Exploration

## What This Notebook Does
Connect to the SQLite database using PySpark,
explore the schema, understand table relationships,
and write exploratory SQL queries.

## Why PySpark Instead of Pandas
Our database has 33+ million transaction rows.
Pandas loads everything into RAM — on a laptop with
8-16GB RAM, loading 33 million rows would be slow
or crash the kernel.

PySpark uses LAZY evaluation — it builds a query plan
but does NOT load data into memory until you explicitly
ask for it. This is the fundamental difference.



In [2]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pyspark.sql import SparkSession
import os

JAR_PATH = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")

DB_PATH = r"D:\some\other\drive\finance_clv.db"
DB_URL = f"jdbc:sqlite:{DB_PATH}"

spark = (
    SparkSession.builder
    .appName("CLV_Database_Setup")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"JAR path      : {JAR_PATH}")
print(f"DB URL        : {DB_URL}")
print("SQLite JDBC driver test:")

spark.sparkContext._jvm.java.lang.Class.forName("org.sqlite.JDBC")

print("SQLite JDBC driver loaded successfully!")

c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version : 4.2.0
JAR path      : c:\Users\dsp96\Desktop\realworld-ds-ml\finance\personal_finance_clv\jars\sqlite-jdbc-3.45.1.0.jar
DB URL        : jdbc:sqlite:D:\some\other\drive\finance_clv.db
SQLite JDBC driver test:
SQLite JDBC driver loaded successfully!


In [4]:
def read_table(table_name):
    """
    Read a table from SQLite into a Spark DataFrame.
    This is the standard pattern we will use throughout
    all Finance notebooks.
    """
    return (
        spark.read
        .format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", table_name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

# Read the customers table
customers = read_table("customers")

print(f"Type: {type(customers)}")
print(f"This is NOT a pandas DataFrame.")
print(f"No data has been loaded into memory yet.")
print(f"Spark has only read the schema from the database.")

Type: <class 'pyspark.sql.classic.dataframe.DataFrame'>
This is NOT a pandas DataFrame.
No data has been loaded into memory yet.
Spark has only read the schema from the database.


In [5]:
def read_table(table_name):
    """
    Standard pattern to read any table from SQLite
    into a Spark DataFrame. Reused across all notebooks.
    """
    return (
        spark.read
        .format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", table_name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

# Read all five tables
customers    = read_table("customers")
products     = read_table("products")
txn_history  = read_table("transactions_history")
txn_label    = read_table("transactions_label")
clv_labels   = read_table("clv_labels")

print("All tables loaded as Spark DataFrames ✅")
print(f"\nObject types:")
print(f"  customers   : {type(customers)}")
print(f"  txn_history : {type(txn_history)}")
print(f"\nNOTE: No data is in memory yet.")
print(f"Spark only read the schema from SQLite.")
print(f"Data loads only when you call .show() or .count()")

All tables loaded as Spark DataFrames ✅

Object types:
  customers   : <class 'pyspark.sql.classic.dataframe.DataFrame'>
  txn_history : <class 'pyspark.sql.classic.dataframe.DataFrame'>

NOTE: No data is in memory yet.
Spark only read the schema from SQLite.
Data loads only when you call .show() or .count()


In [6]:
print("=" * 55)
print(" SCHEMA INSPECTION — All 5 Tables")
print("=" * 55)

tables = {
    "customers"           : customers,
    "products"            : products,
    "transactions_history": txn_history,
    "transactions_label"  : txn_label,
    "clv_labels"          : clv_labels,
}

for name, df in tables.items():
    print(f"\n── {name.upper()} ──────────────────────────")
    df.printSchema()

 SCHEMA INSPECTION — All 5 Tables

── CUSTOMERS ──────────────────────────
root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_tier: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- monthly_income: double (nullable = true)
 |-- account_open_date: string (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- kyc_complete: integer (nullable = true)
 |-- pan_linked: integer (nullable = true)
 |-- aadhaar_linked: integer (nullable = true)
 |-- cibil_score: double (nullable = true)
 |-- digital_score: double (nullable = true)
 |-- rm_assigned: integer (nullable = true)
 |-- is_nri: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)


── PRODUCTS ──────────────────────────
root
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullabl

In [7]:
print("=" * 55)
print(" ROW COUNTS — First Real Spark Action")
print("=" * 55)
print("(This triggers actual data reading for the first time)\n")

for name, df in tables.items():
    count = df.count()
    print(f"  {name:<30}: {count:>12,} rows")

print(f"\nTotal rows across all tables:")
total = sum(df.count() for df in tables.values())
print(f"  {total:>12,} rows")

 ROW COUNTS — First Real Spark Action
(This triggers actual data reading for the first time)

  customers                     :       50,250 rows
  products                      :      124,865 rows
  transactions_history          :   16,608,362 rows
  transactions_label            :   16,608,069 rows
  clv_labels                    :       50,000 rows

Total rows across all tables:
    33,441,546 rows


In [8]:
print("CUSTOMERS TABLE — First 5 rows")
print("=" * 55)
customers.show(5, truncate=False)

print("\nCLV LABELS TABLE — First 5 rows")
print("=" * 55)
clv_labels.show(5, truncate=False)

print("\nTRANSACTIONS HISTORY — First 5 rows")
print("=" * 55)
txn_history.show(5, truncate=True)

CUSTOMERS TABLE — First 5 rows
+-----------+---+------+-------+---------+---------------+--------------+-----------------+----------------+------------+----------+--------------+-----------+-------------+-----------+------+---------------+-------------+----------------------+
|customer_id|age|gender|city   |city_tier|segment        |monthly_income|account_open_date|account_age_days|kyc_complete|pan_linked|aadhaar_linked|cibil_score|digital_score|rm_assigned|is_nri|occupation     |phone        |email                 |
+-----------+---+------+-------+---------+---------------+--------------+-----------------+----------------+------------+----------+--------------+-----------+-------------+-----------+------+---------------+-------------+----------------------+
|CUST000001 |42 |Male  |Pune   |Tier1    |salaried_senior|370941.36     |2021-10-17       |441             |1           |1         |1             |673.0      |6.5          |1          |1     |Salaried Senior|1043321819   |bbalay@ex

In [9]:
# Register all DataFrames as temporary SQL views
# This allows us to query them using SQL syntax
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
txn_history.createOrReplaceTempView("transactions_history")
txn_label.createOrReplaceTempView("transactions_label")
clv_labels.createOrReplaceTempView("clv_labels")

print("All tables registered as SQL views ✅")
print("You can now query them using spark.sql()\n")

# First SparkSQL query — basic sanity check
result = spark.sql("""
    SELECT
        city_tier,
        COUNT(*)                            AS total_customers,
        ROUND(AVG(monthly_income), 2)       AS avg_monthly_income,
        ROUND(AVG(cibil_score), 1)          AS avg_cibil_score,
        SUM(CASE WHEN kyc_complete = 1
                 THEN 1 ELSE 0 END)        AS kyc_complete_count
    FROM customers
    GROUP BY city_tier
    ORDER BY avg_monthly_income DESC
""")

print("Customer distribution by city tier:")
result.show(truncate=False)

# Second query — CLV distribution by bucket
clv_dist = spark.sql("""
    SELECT
        clv_bucket,
        COUNT(*)                              AS customers,
        ROUND(AVG(clv_next_12months), 2)      AS avg_clv,
        ROUND(MIN(clv_next_12months), 2)      AS min_clv,
        ROUND(MAX(clv_next_12months), 2)      AS max_clv
    FROM clv_labels
    GROUP BY clv_bucket
    ORDER BY avg_clv DESC
""")

print("CLV distribution by bucket:")
clv_dist.show(truncate=False)

All tables registered as SQL views ✅
You can now query them using spark.sql()

Customer distribution by city tier:
+---------+---------------+------------------+---------------+------------------+
|city_tier|total_customers|avg_monthly_income|avg_cibil_score|kyc_complete_count|
+---------+---------------+------------------+---------------+------------------+
|Metro    |17600          |311998.36         |668.3          |15654             |
|Tier1    |15012          |182674.21         |664.1          |13385             |
|Tier2    |12648          |123902.56         |665.8          |11296             |
|Tier3    |4990           |86084.85          |667.8          |4438              |
+---------+---------------+------------------+---------------+------------------+

CLV distribution by bucket:
+----------+---------+---------+---------+--------+
|clv_bucket|customers|avg_clv  |min_clv  |max_clv |
+----------+---------+---------+---------+--------+
|very_high |4260     |334160.13|100004.07|90